# 06 – Métricas do Lote 5 (L5)

ESP-02, ESP-03, ESP-06, QA-05, ADD-05 e ADD-06, segundo `spec/metodologia_metricas.md` (secções 7E e 7F).

- Lê `processed/<sensor>_<ano>.parquet` (notebook 01), **um ficheiro de cada vez e só com as colunas necessárias**, e as tabelas `l1_cal_coverage`, `l1_cal_monthly` e `l1_cal_annual` (notebook 02).
- Por ficheiro guarda as células-dia elegíveis e o ADD-05 em `metrics/_partial_lote5/` e **retoma** se a sessão cair (`FORCE = True` refaz). Por ano junta os sensores, calcula o cell-year (dias com deteção por célula) e ESP-02/03.
- O ESP-06 precisa de todos os anos: consolida-se **por blocos de países**, lendo só as linhas desse bloco de cada ano (não carrega todos os cell-years de uma vez). Se faltar memória, reduza `ESP06_BLOCK` ou ative *High-RAM*.
- QA-05 é só um **proxy global** (dia com algum registo no sensor); não é cobertura real por região.
- Escreve `metrics/l5_*.parquet`, `metrics/_log_lote5.csv` e `metrics/_params_lote5.json`.
- O código vem do repo (`git clone` do GitHub público): `data/pipeline/metrics/`.

**Antes de correr:** o atalho `Sprinklers` tem de estar em *O Meu Drive* (ver `data/README.md`) e os notebooks 01 e 02 têm de ter corrido.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
from pathlib import Path

ROOT = Path('/content/drive/MyDrive/Sprinklers')        # adjust if the Drive shortcut has a different name
REPO_URL = 'https://github.com/Rui-Martins31/NASA_Space_Apps_2026_Sprinklers.git'
REPO = Path('/content/repo')
REPO_BRANCH = 'metricas-pipeline'


def ensure_repo():
    if (REPO / '.git').exists():
        subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', REPO_BRANCH], check=True)
        subprocess.run(['git', '-C', str(REPO), 'switch', REPO_BRANCH], check=True)
        subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO)], check=True)


ensure_repo()
sys.path.insert(0, str(REPO / 'data' / 'pipeline'))

In [ ]:
# For a short run: SENSORS = {'modis'}; YEARS = {2000}
SENSORS = {'modis', 'viirs-jpss1'}  # None selects the same core cohort; S-NPP uses notebook 09
YEARS = None            # None selects all available years of the selected sensors
FORCE = False           # Rebuild selected caches even when their provenance matches
GRID_DEG = 0.1
MAX_UNKNOWN_DAYS = 0
ESP06_BLOCK = 40

import subprocess
from pathlib import Path

PROCESSED = ROOT / 'processed'
METRICS = ROOT / 'metrics'
assert PROCESSED.exists(), f"Nao encontro {PROCESSED}. Corra primeiro o notebook 01 e confirme o atalho 'Sprinklers'."
REQUIRED = ['l1_cal_coverage', 'l1_cal_monthly', 'l1_cal_annual']
missing = [n for n in REQUIRED if not (METRICS / f'{n}.parquet').exists()]
if missing:
    raise FileNotFoundError(f'Faltam em {METRICS}: {missing}. Corra primeiro o notebook 02_metrics_lote1.')
PART = METRICS / '_partial_lote5'
PART.mkdir(parents=True, exist_ok=True)

In [ ]:
import json, re, time, datetime
import pandas as pd

from metrics import common
from metrics.calendar_metrics import eligible_cell_days
from metrics.cell_recurrence import build_cell_year, esp02_active_cells, esp03_recurrence, esp06_interannual
from metrics.filter_sensitivity import add05_type_sensitivity
from metrics.coverage_steps import add06_union_step, qa05_confirmed_days

COLS = ['country', 'instrument', 'satellite', 'latitude', 'longitude', 'acq_date', 'type', 'confidence_class']
FILE_RE = re.compile(r'^(?P<sensor>modis|viirs-[a-z0-9]+)_(?P<year>\d{4})\.parquet$', re.I)
CORE_SENSORS = {'modis', 'viirs-jpss1'}
CACHE_SCHEMA = 2


def atomic_write(df, path):
    tmp = path.with_name(path.name + '.tmp')
    df.to_parquet(tmp, index=False)
    tmp.replace(path)


def atomic_json(info, path):
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(info, indent=1))
    tmp.replace(path)


def read_json(path):
    try:
        return json.loads(path.read_text())
    except (OSError, ValueError):
        return {}


def source_signature(sensor, year, path):
    stat = path.stat()
    return dict(schema=CACHE_SCHEMA, sensor=sensor, year=year, path=str(path.resolve()),
                size=stat.st_size, mtime_ns=stat.st_mtime_ns, grid_deg=GRID_DEG,
                confidence=list(common.ELIGIBLE_CONFIDENCE), eligible_type=common.ELIGIBLE_TYPE)


def is_done(sensor, year):
    out = PART / f'{sensor}_{year}'
    return (read_json(out / '_done.json').get('signature') == source_signature(sensor, year, files[(sensor, year)])
            and all((out / f'{name}.parquet').exists() for name in ['cell_days', 'add05']))


def process_file(sensor, year, path):
    """Write only this selected source; commit the provenance marker last."""
    out = PART / f'{sensor}_{year}'
    out.mkdir(exist_ok=True)
    # An invalid marker must not advertise a half-written cache as complete.
    atomic_json({}, out / '_done.json')
    t0 = time.time()
    df = pd.read_parquet(path, columns=COLS)
    expected = 'MODIS' if sensor == 'modis' else 'VIIRS'
    if not df['instrument'].astype(str).str.upper().eq(expected).all():
        raise ValueError(f'{path.name}: instrument does not match {sensor}')
    if sensor == 'viirs-jpss1':
        satellite = df['satellite'].astype(str).str.upper().str.replace(r'[^A-Z0-9]', '', regex=True)
        if not satellite.isin({'N20', 'NOAA20', 'J1', '1'}).all():
            raise ValueError(f'{path.name}: expected NOAA-20; S-NPP belongs in 09_har12_snpp.ipynb')
    info = dict(sensor=sensor, year=year, grid_deg=GRID_DEG, n_rows=len(df),
                signature=source_signature(sensor, year, path))
    atomic_write(eligible_cell_days(df, GRID_DEG), out / 'cell_days.parquet')
    atomic_write(add05_type_sensitivity(df, grid_deg=GRID_DEG), out / 'add05.parquet')
    info['seconds'] = round(time.time() - t0, 1)
    atomic_json(info, out / '_done.json')
    return info


def year_status(year):
    """Only current sources count; combined requires every available core source in this year."""
    every = {s for (s, y) in files if y == year}
    done = [s for s, y in todo if y == year and is_done(s, y)]
    series = set()
    if 'modis' in done:
        series.add('modis')
    if 'viirs-jpss1' in done:
        series.add('viirs')
    if every and set(done) == every:
        series.add('combined')
    return done, series


In [ ]:
selected_sensors = CORE_SENSORS if SENSORS is None else set(SENSORS)
unsupported = selected_sensors - CORE_SENSORS
if unsupported:
    raise ValueError(f'nenhum ficheiro selecionado: unsupported sensors {sorted(unsupported)}; '
                     f'core sensors: {sorted(CORE_SENSORS)}. Use 09_har12_snpp.ipynb for S-NPP.')
files = {}
for p in sorted(PROCESSED.glob('*.parquet')):
    m = FILE_RE.match(p.name)
    if m and m['sensor'].lower() in CORE_SENSORS:
        files[(m['sensor'].lower(), int(m['year']))] = p
todo = sorted(k for k in files if k[0] in selected_sensors and (YEARS is None or k[1] in YEARS))
print(f'{len(files)} core files in processed/, {len(todo)} selected')
if not todo:
    raise ValueError(f'nenhum ficheiro selecionado (SENSORS={SENSORS}, YEARS={YEARS}); '
                     f'available core files in processed/: {sorted(files)}')
years = sorted({y for _, y in todo})
if YEARS is not None and set(YEARS) - set(years):
    raise FileNotFoundError(f'No selected source for years {sorted(set(YEARS) - set(years))}')
selected_dirs = [PART / f'{s}_{y}' for s, y in todo]

for sensor, year in todo:
    if is_done(sensor, year) and not FORCE:
        print(f'[skip] {sensor} {year}')
        continue
    print(f'[process] {sensor} {year}')
    print('   ', process_file(sensor, year, files[(sensor, year)]))

# An annual cache belongs to its exact source cohort, even if other cached sources are newer.
for year in years:
    done, series = year_status(year)
    markers = [PART / f'{s}_{year}' / '_done.json' for s in done]
    targets = [PART / f'cell_year_{year}.parquet', PART / f'esp02_{year}.parquet', PART / f'esp03_{year}.parquet']
    provenance = dict(schema=CACHE_SCHEMA, series=sorted(series),
                      sources=[read_json(m)['signature'] for m in markers])
    meta = PART / f'cell_year_{year}.json'
    if FORCE or read_json(meta) != provenance or not all(t.exists() for t in targets):
        atomic_json({}, meta)
        cd = pd.concat([pd.read_parquet(m.parent / 'cell_days.parquet') for m in markers], ignore_index=True)
        cy = build_cell_year(cd)
        cy = cy[cy['series'].isin(series)].reset_index(drop=True)
        # Persist typed empty years too, replacing stale results without deleting source data.
        atomic_write(cy, targets[0])
        atomic_write(esp02_active_cells(cy), targets[1])
        atomic_write(esp03_recurrence(cy), targets[2])
        atomic_json(provenance, meta)
        print(f'[cell_year] {year}: {len(markers)} sensors, series {sorted(series)}, {len(cy)} rows')


In [ ]:
dirs = selected_dirs
grids = {json.loads((d / '_done.json').read_text()).get('grid_deg') for d in dirs}
assert grids <= {GRID_DEG}, f'parciais calculados com grelhas diferentes {grids}; corra com FORCE = True'


def concat_parts(prefix, keys):
    paths = [PART / f'{prefix}_{year}.parquet' for year in years]
    tbl = pd.concat([pd.read_parquet(p) for p in paths], ignore_index=True)
    tbl = tbl.sort_values(keys).reset_index(drop=True)
    assert not tbl.duplicated(keys).any(), f'{prefix}: duplicate keys across selected years'
    return tbl


tables = {}
tables['l5_add05_type_sensitivity'] = pd.concat(
    [pd.read_parquet(d / 'add05.parquet') for d in dirs], ignore_index=True
).sort_values(['country', 'year', 'instrument', 'variant']).reset_index(drop=True)
assert not tables['l5_add05_type_sensitivity'].duplicated(['country', 'year', 'instrument', 'variant']).any(), \
    'ADD-05 com chaves repetidas entre ficheiros (datas fora do ano do ficheiro?)'
tables['l5_esp02_active_cells'] = concat_parts('esp02', ['country', 'series', 'year'])
tables['l5_esp03_recurrence'] = concat_parts('esp03', ['country', 'series', 'year'])

# ESP-06: anos validos de l1_cal_annual; consolidacao por blocos de paises
annual = pd.read_parquet(METRICS / 'l1_cal_annual.parquet')
ok = (~annual['partial'].astype(bool)) & (annual['n_unknown_days'] <= MAX_UNKNOWN_DAYS)
# so entram (serie, ano) realmente calculados: numa execucao parcial os outros anos nao contam
computed = set()
for year in years:
    computed |= {(s, year) for s in year_status(year)[1]}
comp = pd.DataFrame(sorted(computed), columns=['series', 'year']).astype({'series': 'object', 'year': 'int64'})
eligible_years = annual.loc[ok, ['country', 'series', 'year']].astype({'year': 'int64'}).merge(
    comp, on=['series', 'year']).reset_index(drop=True)
cy_files = [PART / f'cell_year_{year}.parquet' for year in years]
EMPTY_CY = pd.DataFrame({'country': [], 'series': [], 'cell': [], 'year': [], 'n_days': []}).astype(
    {'cell': 'int64', 'year': 'int64', 'n_days': 'int64'})
all_countries = sorted({c for p in cy_files for c in pd.read_parquet(p, columns=['country'])['country'].unique()}
                       | set(eligible_years['country'].unique()))     # inclui paises sem nenhuma celula elegivel
blocks = [all_countries[i:i + ESP06_BLOCK] for i in range(0, len(all_countries), ESP06_BLOCK)] or [[]]
parts06 = []
for block in blocks:
    frames = [pd.read_parquet(p, filters=[('country', 'in', block)]) for p in cy_files]
    cy = pd.concat(frames, ignore_index=True) if frames else EMPTY_CY
    parts06.append(esp06_interannual(cy, eligible_years[eligible_years['country'].isin(block)]))
    del cy, frames
tables['l5_esp06_interannual'] = pd.concat(parts06, ignore_index=True)
tables['l5_esp06_interannual'] = tables['l5_esp06_interannual'].sort_values(['country', 'series']).reset_index(drop=True)

# QA-05 e ADD-06 a partir das tabelas do L1
coverage = pd.read_parquet(METRICS / 'l1_cal_coverage.parquet')
monthly = pd.read_parquet(METRICS / 'l1_cal_monthly.parquet')
instrument_years = {('MODIS' if s == 'modis' else 'VIIRS', y) for s, y in todo}
coverage_years = pd.to_datetime(coverage['date']).dt.year
coverage = coverage.loc[[(i, y) in instrument_years
                         for i, y in zip(coverage['instrument'], coverage_years)]]
monthly = monthly.merge(comp, on=['series', 'year'], how='inner')

tables['l5_qa05_confirmed_days'] = qa05_confirmed_days(coverage)
tables['l5_add06_union_step'] = add06_union_step(monthly, max_unknown_days=MAX_UNKNOWN_DAYS)

for name, tbl in tables.items():
    atomic_write(tbl, METRICS / f'{name}.parquet')
    print(f'{name}: {len(tbl)} linhas')

try:
    commit = subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
except Exception:
    commit = ''
params = dict(grid_deg=GRID_DEG, max_unknown_days=MAX_UNKNOWN_DAYS, esp06_block=ESP06_BLOCK,
              sensors=sorted(selected_sensors), years=years, selected_files=[files[k].name for k in todo],
              series_years_computed=[f'{s}:{y}' for s, y in sorted(computed)],
              repo_commit=commit, run_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'))
(METRICS / '_params_lote5.json').write_text(json.dumps(params, indent=1))
log = pd.DataFrame([dict(table=n, n_rows=len(t), **params) for n, t in tables.items()])
log.to_csv(METRICS / '_log_lote5.csv', index=False)
print(log[['table', 'n_rows']].to_string(index=False))

## Verificação
Confirma invariantes entre as tabelas e contra o L1 (falha com `AssertionError` se alguma não bater).

In [ ]:
e2, e3 = tables['l5_esp02_active_cells'], tables['l5_esp03_recurrence']
keys = ['country', 'series', 'year']
assert e2[keys + ['n_active_cells']].equals(e3[keys + ['n_active_cells']]), 'ESP-02 != n_active_cells do ESP-03'
buckets = e3[['n_cells_1d', 'n_cells_2_5d', 'n_cells_6_15d', 'n_cells_16plus']].sum(axis=1)
assert (buckets == e3['n_active_cells']).all(), 'baldes de recorrencia nao somam n_active_cells'
e6 = tables['l5_esp06_interannual']
assert (e6['n_cells_2plus_years'] <= e6['n_cells']).all()
assert e6['pct_cells_2plus'].dropna().between(0, 100).all()
q5 = tables['l5_qa05_confirmed_days']
assert q5['pct_confirmed'].between(0, 100).all() and (q5['n_days_confirmed'] <= q5['n_days_expected']).all()
a6 = tables['l5_add06_union_step']
assert (a6['ratio_combined_modis'].dropna() >= 1).all(), 'a uniao nunca pode ter menos celulas-dia que o MODIS'

# contra o L1: a soma de n_days por (pais, serie, ano) e a atividade anual do CAL-04
ann = annual.set_index(['country', 'series', 'year'])['cell_days']
checked = 0
for p in cy_files:
    year = int(re.search(r'(\d{4})', p.name).group(1))
    for (country, series), total in pd.read_parquet(p).groupby(['country', 'series'], observed=True)['n_days'].sum().items():
        assert (series, year) in computed, f'{series} {year} no cell-year mas nao calculada por completo'
        assert ann.get((country, series, year)) == total, f'{country} {series} {year}: cell-year != CAL-04'
        checked += 1
print(f'OK: invariantes verificadas ({checked} comparacoes com o CAL-04)')